In [ ]:
import torch
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

In [ ]:
root = "./data"

In [ ]:
# 每次读取图片时，转换为形状 [1, 28, 28] 的张量
transform = transforms.ToTensor()

full_train_dataset = MNIST(
    root=root,
    train=True,
    transform=transform,
    download=False
)

test_dataset = MNIST(
    root=root,
    train=False,
    transform=transform,
    download=False
)

In [ ]:
# 固定随机种子，让每次划分得到相同的训练集和验证集
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [55000, 5000],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
# 训练时打乱顺序；验证和测试保持顺序
# 当前在 Windows 的 Jupyter 中，先使用单进程加载
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=0
)

test_loader = DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=0
)

In [ ]:
# 取出一批数据，检查数量、尺寸和像素范围
images, labels = next(iter(train_loader))

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Pixel range:", images.min().item(), images.max().item())

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
#Sequential 按照顺序连接
model = nn.Sequential(
    nn.Conv2d(1,4,kernel_size=3,stride=1,padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2,stride=2),

    nn.Conv2d(4,8,kernel_size=3,stride=1,padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2,stride=2),
    nn.Flatten(),
    nn.Linear(200,10)
)

In [ ]:
import torch
from torch import nn

# 有可用的 NVIDIA GPU 就使用 GPU，否则使用 CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

# 损失函数：衡量分类结果与正确标签之间的差异
criterion = nn.CrossEntropyLoss()

# 优化器：根据梯度更新权重，学习率为 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model.train()

# 获取一批数据，并移动到 GPU
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

# 清除旧梯度
optimizer.zero_grad(set_to_none=True)

# 前向传播和计算损失
outputs = model(images)
loss = criterion(outputs, labels)

# 反向传播和更新权重
loss.backward()
optimizer.step()

print("Output shape:", outputs.shape)
print("Loss:", loss.item())
print("One training step completed.")

In [ ]:
# 训练一轮：更新权重，并统计损失和准确率
def train_one_epoch(loader):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


# 验证：评估当前权重的效果
def evaluate(loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += batch_size

    return total_loss / total, correct / total

In [ ]:
print("Training started...")

train_loss, train_acc = train_one_epoch(train_loader)
val_loss, val_acc = evaluate(val_loader)

print(f"Train loss: {train_loss:.4f}, accuracy: {train_acc:.2%}")
print(f"Val loss:   {val_loss:.4f}, accuracy: {val_acc:.2%}")

In [ ]:
from pathlib import Path

def save_weights(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("wb") as f:
        torch.save(model.state_dict(), f)

# 测试保存当前权重
save_weights(checkpoint_path)

print("Directory exists:", checkpoint_path.parent.is_dir())
print("Saved file:", checkpoint_path)
print("File size:", checkpoint_path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

checkpoint_path = Path(
    "./checkpoints/mnist_cnn_best.pth"
)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# 先保存已经完成的第1轮，作为比较基准
best_val_acc = val_acc
best_val_loss = val_loss
best_epoch = 1

save_weights(checkpoint_path)

# 每条记录：轮数、训练损失、训练准确率、验证损失、验证准确率
history = [(1, train_loss, train_acc, val_loss, val_acc)]

# 接着训练剩下的19轮
for epoch in range(2, 21):
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss, train_acc = train_one_epoch(train_loader)
    val_loss, val_acc = evaluate(val_loader)

    history.append(
        (epoch, train_loss, train_acc, val_loss, val_acc)
    )

    # 验证效果改善时，更新最佳权重文件
    if (
        val_acc > best_val_acc
        or (val_acc == best_val_acc and val_loss < best_val_loss)
    ):
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch

        save_weights(checkpoint_path)

    print(
        f"Epoch {epoch:02d}/20 | lr={current_lr:.1e} | "
        f"Train loss={train_loss:.4f}, acc={train_acc:.2%} | "
        f"Val loss={val_loss:.4f}, acc={val_acc:.2%}"
    )

    # 下一轮开始使用更小的学习率
    if epoch in (12, 18):
        for group in optimizer.param_groups:
            group["lr"] *= 0.2

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation accuracy: {best_val_acc:.2%}")
print("Saved:", checkpoint_path)


In [ ]:
import torch
from torch import nn
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 保存的是参数，所以加载前需要先定义相同的网络结构
model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    nn.Conv2d(4, 8, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    nn.Flatten(),
    nn.Linear(200, 10)
).to(device)

# 加载训练好的权重
checkpoint_path = Path("./checkpoints/mnist_cnn_best.pth")

with checkpoint_path.open("rb") as f:
    state_dict = torch.load(
        f, map_location=device, weights_only=True
    )

model.load_state_dict(state_dict)
model.eval()

# 加载本地测试集
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transforms.ToTensor(),
    download=False
)

test_loader = DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=0
)

# 评估 FP32 模型
criterion = nn.CrossEntropyLoss()
total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

print("Device:", device)
print(f"FP32 test loss: {total_loss / total:.4f}")
print(f"FP32 test accuracy: {correct / total:.2%}")


In [ ]:
import math

quantized_weights = {}

for name, parameter in model.named_parameters():
    if not name.endswith(".weight"):
        continue

    weight = parameter.detach().cpu()
    max_abs = weight.abs().max().item()

    # 选择能覆盖权重范围的、最小的 2 的整数次幂尺度
    if max_abs == 0:
        scale = 1.0
    else:
        exponent = math.ceil(math.log2(max_abs / 127))
        scale = 2.0 ** exponent

    # 浮点权重 → INT8
    weight_int8 = torch.round(weight / scale)
    weight_int8 = weight_int8.clamp(-127, 127).to(torch.int8)

    # 还原为浮点近似值，观察量化误差
    restored_weight = weight_int8.float() * scale
    max_error = (weight - restored_weight).abs().max().item()

    quantized_weights[name] = {
        "weight_int8": weight_int8,
        "scale": scale
    }

    print(
        f"{name} | shape={tuple(weight.shape)} | "
        f"scale={scale:.8f} | max_error={max_error:.8f}"
    )

In [ ]:
from copy import deepcopy
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# 按训练时相同的划分方式，恢复 5000 张验证图片
full_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transforms.ToTensor(),
    download=False
)

_, val_dataset = random_split(
    full_train_dataset,
    [55000, 5000],
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=0
)

# 复制模型，使用“整数权重 × 尺度”替换副本中的权重
weight_quant_model = deepcopy(model)

with torch.no_grad():
    for name, parameter in weight_quant_model.named_parameters():
        if name in quantized_weights:
            info = quantized_weights[name]
            restored = info["weight_int8"].float() * info["scale"]
            parameter.copy_(restored.to(parameter.device))

def validation_accuracy(net, loader):
    net.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            predictions = net(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total

fp32_val_acc = validation_accuracy(model, val_loader)
weight_quant_val_acc = validation_accuracy(weight_quant_model, val_loader)

print(f"FP32 validation accuracy: {fp32_val_acc:.2%}")
print(f"Weight-quantized validation accuracy: {weight_quant_val_acc:.2%}")
print(
    f"Accuracy drop: "
    f"{(fp32_val_acc - weight_quant_val_acc) * 100:.2f} percentage points"
)

In [ ]:
import math
from torch.utils.data import DataLoader, Subset, random_split

# 恢复原来的训练集划分
calibration_train_dataset, _ = random_split(
    full_train_dataset,
    [55000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# 从训练集随机选取 1024 张图片
indices = torch.randperm(
    len(calibration_train_dataset),
    generator=torch.Generator().manual_seed(123)
)[:1024].tolist()

calibration_dataset = Subset(calibration_train_dataset, indices)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

max_abs = {"input": 0.0, "pool1": 0.0, "pool2": 0.0}

# 使用已经替换为量化近似权重的模型收集范围
weight_quant_model.eval()

with torch.no_grad():
    for images, _ in calibration_loader:
        x = images.to(device)
        max_abs["input"] = max(
            max_abs["input"], x.abs().max().item()
        )

        # 第一层卷积 → ReLU → 最大池化
        x = weight_quant_model[:3](x)
        max_abs["pool1"] = max(
            max_abs["pool1"], x.abs().max().item()
        )

        # 第二层卷积 → ReLU → 最大池化
        x = weight_quant_model[3:6](x)
        max_abs["pool2"] = max(
            max_abs["pool2"], x.abs().max().item()
        )

# 为三个位置计算 INT8 量化尺度
activation_scales = {}

for name, maximum in max_abs.items():
    if maximum == 0:
        scale = 1.0
    else:
        scale = 2.0 ** math.ceil(math.log2(maximum / 127))

    activation_scales[name] = scale

    print(
        f"{name} | max_abs={maximum:.6f} | "
        f"scale={scale:.8f}"
    )

In [ ]:
import torch
import math

# 层名称、模型中的编号、输入尺度名称、输出尺度名称
configs = [
    ("conv1", "0", "input", "pool1"),
    ("conv2", "3", "pool1", "pool2"),
    ("fc",    "7", "pool2", None)
]

parameters = dict(model.named_parameters())
integer_layers = {}

for layer_name, prefix, input_key, output_key in configs:
    weight_info = quantized_weights[f"{prefix}.weight"]

    input_scale = activation_scales[input_key]
    weight_scale = weight_info["scale"]
    accumulator_scale = input_scale * weight_scale

    # 偏置与乘加结果使用相同的尺度
    bias = parameters[f"{prefix}.bias"].detach().cpu().double()
    rounded_bias = torch.round(bias / accumulator_scale)

    # 转换前检查 INT32 范围
    limits = torch.iinfo(torch.int32)
    if (
        (rounded_bias < limits.min) |
        (rounded_bias > limits.max)
    ).any().item():
        raise OverflowError(f"{layer_name}: bias exceeds INT32 range")

    bias_int32 = rounded_bias.to(torch.int32)

    if output_key is None:
        # 全连接输出保留为 INT32
        output_scale = accumulator_scale
        output_shift = None
    else:
        output_scale = activation_scales[output_key]
        output_shift = int(round(math.log2(
            output_scale / accumulator_scale
        )))

    integer_layers[layer_name] = {
        "weight_int8": weight_info["weight_int8"],
        "bias_int32": bias_int32,
        "input_scale": input_scale,
        "weight_scale": weight_scale,
        "accumulator_scale": accumulator_scale,
        "output_scale": output_scale,
        "output_shift": output_shift
    }

    shift_text = (
        "keep INT32" if output_shift is None
        else str(output_shift)
    )

    print(
        f"{layer_name} | bias_dtype={bias_int32.dtype} | "
        f"bias_scale={accumulator_scale:.8f} | "
        f"output_shift={shift_text}"
    )

In [ ]:
from mnist_int8_pipeline import run_quantized_evaluation

int8_model, report = run_quantized_evaluation(
    model=model,
    quantized_weights=quantized_weights,
    activation_scales=activation_scales,
    val_loader=val_loader,
    test_loader=test_loader,
    fp32_val_acc=0.9776,
    fp32_test_acc=0.9818,
    max_drop_pp=1.0,
    min_test_acc=0.96,
    output_path="./quantized/mnist_cnn_int8.pth",
)

In [ ]:
from mnist_int8_pipeline import IntegerMnistCNN

saved_int8_model = IntegerMnistCNN.from_checkpoint(
    "./quantized/mnist_cnn_int8.pth"
)

images, labels = next(iter(test_loader))
logits = saved_int8_model.forward(images)
predictions = logits.argmax(axis=1)

print("Logits shape:", logits.shape)
print("Logits dtype:", logits.dtype)
print("Predictions:", predictions[:10])
print("Labels:     ", labels[:10].numpy())

In [ ]:
import json
from pathlib import Path
import numpy as np

export_dir = Path("./fpga_vectors")
export_dir.mkdir(parents=True, exist_ok=True)

def write_hex(filename, values, bits):
    """每行写一个定宽二进制补码数，供 $readmemh 读取。"""
    digits = bits // 4
    mask = (1 << bits) - 1

    with (export_dir / filename).open("w", encoding="ascii") as f:
        for value in np.asarray(values).reshape(-1):
            f.write(f"{int(value) & mask:0{digits}x}\n")

package = saved_int8_model.package

# 导出三层权重和偏置
layer_metadata = {}

for name in ("conv1", "conv2", "fc"):
    layer = package["layers"][name]
    weights = layer["weight_int8"].numpy()
    biases = layer["bias_int32"].numpy()

    write_hex(f"{name}_weights.hex", weights, bits=8)
    write_hex(f"{name}_bias.hex", biases, bits=32)

    layer_metadata[name] = {
        "weight_shape": list(weights.shape),
        "input_scale": layer["input_scale"],
        "weight_scale": layer["weight_scale"],
        "bias_scale": layer["accumulator_scale"],
        "output_scale": layer["output_scale"],
        "output_shift": layer["output_shift"],
        "accumulator_abs_bound": layer["accumulator_abs_bound"],
    }

# 导出一张图片的量化输入和对应整数 logits
images, labels = next(iter(test_loader))
input_int8 = saved_int8_model.quantize_input(images[:1])
logits_int32 = saved_int8_model.forward_quantized(input_int8)[0]

write_hex("input_0.hex", input_int8, bits=8)
write_hex("logits_0.hex", logits_int32, bits=32)

metadata = {
    "input_shape": [1, 28, 28],
    "input_scale": package["input_scale"],
    "input_layout": "NCHW; channel-row-column order",
    "conv_weight_layout": "OIHW; each 3x3 kernel is row-major",
    "fc_weight_layout": "class-major; input features use channel-row-column order",
    "rounding": package["rounding"],
    "activation_range": package["activation_range"],
    "layers": layer_metadata,
    "golden_sample": {
        "label": int(labels[0]),
        "prediction": int(logits_int32.argmax()),
        "logits_int32": [int(x) for x in logits_int32],
    },
}

(export_dir / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Exported to:", export_dir.resolve())
print("Golden label:", int(labels[0]))
print("Golden prediction:", int(logits_int32.argmax()))